Fase 5 Modul 3: Window Function

In [1]:
from pathlib import Path
import pandas as pd, sqlite3
csv = Path("data/clean_tips.csv") if Path("data/clean_tips.csv").exists() else Path("../data/clean_tips.csv")
con = sqlite3.connect(csv.parent / "tips.db")

# 1. Peringkat per grup: top 2 tip terbesar per hari (ROW_NUMBER). PARTITION = grup, ORDER = urutan.
print(pd.read_sql("""SELECT day, total_bayar, tip,
ROW_NUMBER() OVER (PARTITION BY day ORDER BY tip DESC) AS peringkat
FROM tips LIMIT 6""", con))

# 2. Bandingkan dengan baris lain: tip vs transaksi sebelumnya (LAG). Butuh ORDER yang jelas.
print(pd.read_sql("""SELECT day, total_bayar,
LAG(total_bayar) OVER (ORDER BY total_bayar) AS bill_sebelumnya
FROM tips LIMIT 4""", con))

# 3. Total berjalan: omzet kumulatif dari tagihan terkecil ke terbesar.
print(pd.read_sql("""SELECT total_bayar,
SUM(total_bayar) OVER (ORDER BY total_bayar) AS omzet_kumulatif
FROM tips LIMIT 4""", con))

   day  total_bayar   tip  peringkat
0  Fri        44.90  4.73          1
1  Fri        20.62  4.30          2
2  Fri        31.28  4.00          3
3  Fri        25.99  3.50          4
4  Fri        16.90  3.48          5
5  Fri        26.00  3.25          6
    day  total_bayar  bill_sebelumnya
0   Sat         4.07              NaN
1   Fri         6.75             4.07
2   Sat         8.25             6.75
3  Thur         9.00             8.25
   total_bayar  omzet_kumulatif
0         4.07             4.07
1         6.75            10.82
2         8.25            19.07
3         9.00            28.07


In [2]:
# ROW_NUMBER: 1,2,3,4 selalu unik. RANK: nilai sama = peringkat sama, lalu lompat (1,1,3). DENSE_RANK: sama tapi tidak lompat (1,1,2).
print(pd.read_sql("""SELECT tip,
ROW_NUMBER() OVER (ORDER BY tip DESC) AS rn,
RANK() OVER (ORDER BY tip DESC) AS rnk,
DENSE_RANK() OVER (ORDER BY tip DESC) AS drnk
FROM tips LIMIT 6""", con))
# Aturan: butuh tepat N baris (top 3) -> ROW_NUMBER. Peringkat lomba (juara 1 kembar) -> RANK.

     tip  rn  rnk  drnk
0  10.00   1    1     1
1   9.00   2    2     2
2   7.58   3    3     3
3   6.73   4    4     4
4   6.70   5    5     5
5   6.50   6    6     6
